In [ ]:
!pip install rasterio geopandas matplotlib numpy scikit-image tqdm glob2
!pip install --upgrade --force-reinstall shapely

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

data_dir = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second'
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"❌ Folder not found: {data_dir}")
else:
    print(f"✅ Data folder located: {data_dir}")


In [ ]:
tif_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.tif')]
tif_files.sort()

print(f"✅ Found {len(tif_files)} TIFs:")
for f in tif_files:
    print(os.path.basename(f))


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

def show_truecolor(src, title="RGB Visualization"):
    # Sentinel-2: RGB = Bands 4,3,2
    r = src.read(3).astype('float32')
    g = src.read(2).astype('float32')
    b = src.read(1).astype('float32')
    rgb = np.dstack((r, g, b))
    rgb = np.clip(rgb / np.percentile(rgb, 98), 0, 1)
    plt.figure(figsize=(5,5))
    plt.imshow(rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

for tif in tqdm(tif_files, desc="Displaying RGB composites"):
    with rasterio.open(tif) as src:
        show_truecolor(src, os.path.basename(tif))


In [ ]:
def compute_indices(src):
    """Return NDWI and MNDWI arrays."""
    green = src.read(2).astype('float32')   # Band 3
    nir   = src.read(4).astype('float32')   # Band 8
    swir1 = src.read(6).astype('float32')   # Band 11

    ndwi  = (green - nir) / (green + nir + 1e-6)
    mndwi = (green - swir1) / (green + swir1 + 1e-6)
    return np.clip(ndwi, -1, 1), np.clip(mndwi, -1, 1)

# Preview NDWI & MNDWI for one image
with rasterio.open(tif_files[0]) as src:
    ndwi, mndwi = compute_indices(src)

fig, axs = plt.subplots(1, 2, figsize=(10,5))
axs[0].imshow(ndwi, cmap='BrBG'); axs[0].set_title('NDWI')
axs[1].imshow(mndwi, cmap='BrBG'); axs[1].set_title('MNDWI')
for ax in axs: ax.axis('off')
plt.suptitle(os.path.basename(tif_files[0]))
plt.show()


In [ ]:
save_dir = os.path.join(data_dir, "Water_Indices")
os.makedirs(save_dir, exist_ok=True)

for tif in tqdm(tif_files, desc="Computing NDWI & MNDWI"):
    name = os.path.splitext(os.path.basename(tif))[0]
    with rasterio.open(tif) as src:
        ndwi, mndwi = compute_indices(src)
        meta = src.meta.copy()
        meta.update(count=2, dtype='float32')

        out_path = os.path.join(save_dir, f"{name}_NDWI_MNDWI.tif")
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(ndwi, 1)
            dst.write(mndwi, 2)

print(f"✅ All NDWI & MNDWI rasters saved in:\n{save_dir}")


In [ ]:
from glob import glob
ndwi_files = glob(os.path.join(save_dir, "*_NDWI_MNDWI.tif"))
ndwi_files.sort()

for f in tqdm(ndwi_files, desc="Visualizing NDWI & MNDWI"):
    with rasterio.open(f) as src:
        ndwi = src.read(1)
        mndwi = src.read(2)
        fig, axs = plt.subplots(1,2,figsize=(10,5))
        axs[0].imshow(ndwi, cmap='BrBG'); axs[0].set_title('NDWI')
        axs[1].imshow(mndwi, cmap='BrBG'); axs[1].set_title('MNDWI')
        for ax in axs: ax.axis('off')
        plt.suptitle(os.path.basename(f))
        plt.show()


Water Bodies Extraction

In [ ]:
import os
import glob
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.plot import show
import pandas as pd

In [ ]:
# ======================================================
# 1️⃣ DEFINE FOLDER PATH (Already exported from GEE)
# ======================================================
waterindices_folder = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Water_Indices"

# ======================================================
# 2️⃣ LOAD ALL GEO-TIFF FILES
# ======================================================
tif_files = [os.path.join(waterindices_folder, f) for f in os.listdir(waterindices_folder) if f.endswith('.tif')]
tif_files.sort()

print(f"✅ Found {len(tif_files)} NDWI/MNDWI GeoTIFFs for analysis.")

In [ ]:
# ======================================================
# 3️⃣ FUNCTION: EXTRACT WATER MASK (NDWI/MNDWI > 0)
# ======================================================
def extract_water_mask(img_array, threshold=0):
    """
    Create binary mask for water pixels using NDWI/MNDWI > threshold.
    """
    return np.where(img_array > threshold, 1, 0)

# ======================================================
# 4️⃣ FUNCTION: COMPUTE WATER AREA (km²)
# ======================================================
def compute_water_area(mask, transform):
    """
    Compute area (km²) of water pixels from binary mask.
    """
    pixel_area = abs(transform[0] * transform[4])  # m² per pixel
    total_water_pixels = np.sum(mask == 1)
    return (total_water_pixels * pixel_area) / 1e6  # convert to km²

# ======================================================
# 5️⃣ PROCESS EACH FILE
# ======================================================
results = []

for tif in tif_files:
    with rasterio.open(tif) as src:
        img = src.read(1)  # assuming NDWI/MNDWI is single band
        transform = src.transform
        name = os.path.basename(tif)

        # Handle invalid pixels
        img = np.where(np.isnan(img), -9999, img)

        # Water mask and area
        mask = extract_water_mask(img, threshold=0)
        area_km2 = compute_water_area(mask, transform)
        results.append((name, area_km2))

        # Visualization
        fig, axs = plt.subplots(1, 3, figsize=(16, 6))
        show(img, ax=axs[0], cmap='viridis', title=f"{name} — NDWI/MNDWI")
        axs[1].imshow(mask, cmap='Blues')
        axs[1].set_title("Detected Water Mask (>0)")
        axs[1].axis('off')

        # Overlay comparison
        axs[2].imshow(img, cmap='gray')
        axs[2].imshow(mask, cmap='Blues', alpha=0.4)
        axs[2].set_title("Overlay (Water highlighted)")
        axs[2].axis('off')

        plt.suptitle(f"💧 {name} | Water Area: {area_km2:.2f} km²", fontsize=14)
        plt.tight_layout()
        plt.show()

# ======================================================
# 6️⃣ CREATE SUMMARY DATAFRAME
# ======================================================
import pandas as pd

df_water = pd.DataFrame(results, columns=['Filename', 'Water Area (km²)'])
df_water['Year'] = df_water['Filename'].str.extract(r'(\d{4})')
df_water['Season'] = df_water['Filename'].str.extract(r'_(\d{2}_[A-Za-z]+)')

print("✅ Water body extraction complete!\n")
display(df_water)

# ======================================================
# 7️⃣ VISUALIZE WATER AREA OVER TIME
# ======================================================
plt.figure(figsize=(12,6))
plt.bar(df_water['Filename'], df_water['Water Area (km²)'])
plt.xticks(rotation=90)
plt.title("🌊 Water Body Area (NDWI/MNDWI > 0) Across All Exports", fontsize=14)
plt.ylabel("Water Area (km²)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ======================================================
# 8️⃣ SAVE WATER MASKS AS RASTERS + SHAPEFILES
# ======================================================
import os
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape   # ✅ Fix: re-import here

# Output directories
raster_output_dir = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Water_Extracted_Rasters"
vector_output_dir = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Water_Extracted_Shapes"

os.makedirs(raster_output_dir, exist_ok=True)
os.makedirs(vector_output_dir, exist_ok=True)

print("💾 Output folders ready:")
print(" ├─ Rasters →", raster_output_dir)
print(" └─ Shapefiles →", vector_output_dir)

# ======================================================
# 9️⃣ LOOP THROUGH NDWI/MNDWI FILES AND SAVE RESULTS
# ======================================================

for tif in tif_files:
    with rasterio.open(tif) as src:
        img = src.read(1)
        transform = src.transform
        crs = src.crs
        name = os.path.basename(tif).replace(".tif", "")

        img = np.where(np.isnan(img), -9999, img)
        mask = extract_water_mask(img, threshold=0)

        # ✅ Save rasterized binary water mask
        out_raster_path = os.path.join(raster_output_dir, f"{name}_WATERMASK.tif")
        meta = src.meta.copy()
        meta.update({"count": 1, "dtype": "uint8"})
        with rasterio.open(out_raster_path, "w", **meta) as dst:
            dst.write(mask.astype("uint8"), 1)

        # ✅ Convert to polygons (vectorize)
        mask_for_shapes = mask.astype("uint8")
        shapes_gen = shapes(mask_for_shapes, mask=mask_for_shapes == 1, transform=transform)
        polygons = [shape(geom) for geom, val in shapes_gen if val == 1]

        if len(polygons) > 0:
            gdf = gpd.GeoDataFrame(geometry=polygons, crs=crs)
            out_vector_path = os.path.join(vector_output_dir, f"{name}_WATERMASK.shp")
            gdf.to_file(out_vector_path)
            print(f"✅ Saved {name} → Raster + Shapefile")
        else:
            print(f"⚠️ No water polygons detected for {name}")

print("\n🎉 All water bodies extracted, vectorized, and saved to Drive!")


Rivers and Reservoirs Extraction

In [ ]:
import os
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.affinity import scale
from shapely.geometry import LineString
import warnings

# === Directory setup ===
parent_dir = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second"
vector_dir = os.path.join(parent_dir, "Water_Extracted_Shapes")
river_dir = os.path.join(parent_dir, "Rivers")
reservoir_dir = os.path.join(parent_dir, "Reservoirs")

os.makedirs(river_dir, exist_ok=True)
os.makedirs(reservoir_dir, exist_ok=True)

# === NEW HELPER: Flatten Geometries ===
def flatten_geometries(geom):
    """
    Takes a geometry and returns a list of *only* Polygons,
    handling MultiPolygons and GeometryCollections.
    """
    if geom.is_empty:
        return []
    if geom.geom_type == 'Polygon':
        return [geom]
    if geom.geom_type == 'MultiPolygon':
        return list(geom.geoms)
    if geom.geom_type == 'GeometryCollection':
        polygons = []
        for g in geom.geoms:
            polygons.extend(flatten_geometries(g))
        return polygons
    # Ignore points and lines
    return []

# === Main Loop ===

# --- TUNED PARAMETERS ---
# 1. "More Lenient on Reservoir": Smaller bite (300m)
#    Only channels < 600m wide will be classified as rivers.
EROSION_DISTANCE_METERS = 300

# 2. "More Strict on River": Aggressive noise filter (1 km^2)
#    Ignores all water bodies smaller than 1 square kilometer.
MIN_AREA_THRESHOLD_METERS = 1000000  # (1.0 km^2)
# --- END TUNED PARAMETERS ---

# Suppress warnings from GeoPandas about CRS
warnings.filterwarnings('ignore', 'GeoSeries.isna', UserWarning)

for file in os.listdir(vector_dir):
    if not file.endswith(".shp"):
        continue

    shp_path = os.path.join(vector_dir, file)
    try:
        gdf = gpd.read_file(shp_path)
    except Exception as e:
        print(f"⚠️ {file}: Could not read file, skipping. Error: {e}")
        continue

    # --- 1. Pre-processing ---
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf.geometry = gdf.geometry.buffer(0)  # Fix invalid geometries

    # Calculate area and apply our NEW, STRICT threshold
    gdf['area'] = gdf.geometry.area
    gdf = gdf[gdf['area'] > MIN_AREA_THRESHOLD_METERS].copy()

    if gdf.empty:
        print(f"⚠️ {file}: No polygons found larger than 1.0 km^2, skipping.")
        continue

    all_rivers = []
    all_reservoirs = []

    # --- 2. Morphological Separation (Erode & Dilate) ---
    for geom in gdf.geometry:
        # 1. ERODE (Buffer -N)
        # Apply our new, smaller 300m "bite"
        reservoir_cores = geom.buffer(-EROSION_DISTANCE_METERS, cap_style=3, join_style=2)

        if reservoir_cores.is_empty:
            # This is now a "Significant River":
            # It was > 1 km^2 in area, but < 600m wide.
            all_rivers.extend(flatten_geometries(geom))
            continue

        # 3. DILATE (Buffer +N)
        reservoir_geoms = reservoir_cores.buffer(EROSION_DISTANCE_METERS, cap_style=3, join_style=2)

        # 4. SUBTRACT to find the rivers
        river_geoms = geom.difference(reservoir_geoms)

        # 5. Add results
        all_reservoirs.extend(flatten_geometries(reservoir_geoms))
        all_rivers.extend(flatten_geometries(river_geoms))

    # --- 3. Save outputs ---
    rivers_gdf = gpd.GeoDataFrame(geometry=all_rivers, crs=gdf.crs)
    reservoirs_gdf = gpd.GeoDataFrame(geometry=all_reservoirs, crs=gdf.crs)

    # Final cleanup of any tiny slivers created by the 'difference'
    rivers_gdf = rivers_gdf[rivers_gdf.geometry.area > MIN_AREA_THRESHOLD_METERS]
    reservoirs_gdf = reservoirs_gdf[reservoirs_gdf.geometry.area > MIN_AREA_THRESHOLD_METERS]

    rivers_path = os.path.join(river_dir, f"rivers_{file}")
    reservoirs_path = os.path.join(reservoir_dir, f"reservoirs_{file}")

    if not rivers_gdf.empty:
        rivers_gdf.to_file(rivers_path)
    if not reservoirs_gdf.empty:
        reservoirs_gdf.to_file(reservoirs_path)

    print(f"✅ {file}: {len(rivers_gdf)} rivers, {len(reservoirs_gdf)} reservoirs extracted.")

print("🎯 Extraction complete — Rivers and Reservoirs saved in their respective folders.")

New method for river and reservoir extraction

In [ ]:
import os
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.affinity import scale
from shapely.geometry import LineString
import warnings

# === Directory setup (Updated Folders) ===
parent_dir = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second"
vector_dir = os.path.join(parent_dir, "Water_Extracted_Shapes")
# --- NEW FOLDER NAMES ---
river_dir = os.path.join(parent_dir, "Rivers2")
reservoir_dir = os.path.join(parent_dir, "Reservoirs2")

os.makedirs(river_dir, exist_ok=True)
os.makedirs(reservoir_dir, exist_ok=True)

# === NEW HELPER: Flatten Geometries ===
def flatten_geometries(geom):
    """
    Takes a geometry and returns a list of *only* Polygons,
    handling MultiPolygons and GeometryCollections.
    """
    if geom.is_empty:
        return []
    if geom.geom_type == 'Polygon':
        return [geom]
    if geom.geom_type == 'MultiPolygon':
        return list(geom.geoms)
    if geom.geom_type == 'GeometryCollection':
        polygons = []
        for g in geom.geoms:
            polygons.extend(flatten_geometries(g))
        return polygons
    # Ignore points and lines
    return []

# === Main Loop ===

# --- TUNED PARAMETERS ---
# 1. "More Lenient on Reservoir": Keep 300m "bite"
#    Only channels < 600m wide will be classified as rivers.
EROSION_DISTANCE_METERS = 300

# 2. "Balanced Filter": (THE FIX)
#    1.0 km^2 was too high and deleted the main river.
#    0.1 km^2 was too low and kept 200+ speckles.
#    Let's try 0.25 km^2 as a balance.
MIN_AREA_THRESHOLD_METERS = 250000  # (0.25 km^2)
# --- END TUNED PARAMETERS ---

# Suppress warnings
warnings.filterwarnings('ignore', 'GeoSeries.isna', UserWarning)

for file in os.listdir(vector_dir):
    if not file.endswith(".shp"):
        continue

    shp_path = os.path.join(vector_dir, file)
    try:
        gdf = gpd.read_file(shp_path)
    except Exception as e:
        print(f"⚠️ {file}: Could not read file, skipping. Error: {e}")
        continue

    # --- 1. Pre-processing ---
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    gdf.geometry = gdf.geometry.buffer(0)  # Fix invalid geometries

    # Calculate area and apply our NEW, BALANCED threshold
    gdf['area'] = gdf.geometry.area
    gdf = gdf[gdf['area'] > MIN_AREA_THRESHOLD_METERS].copy()

    if gdf.empty:
        print(f"⚠️ {file}: No polygons found larger than 0.25 km^2, skipping.")
        continue

    all_rivers = []
    all_reservoirs = []

    # --- 2. Morphological Separation (Erode & Dilate) ---
    for geom in gdf.geometry:
        # 1. ERODE (Buffer -N)
        reservoir_cores = geom.buffer(-EROSION_DISTANCE_METERS, cap_style=3, join_style=2)

        if reservoir_cores.is_empty:
            # This is a "Significant River":
            # > 0.25 km^2 in area, but < 600m wide.
            all_rivers.extend(flatten_geometries(geom))
            continue

        # 3. DILATE (Buffer +N)
        reservoir_geoms = reservoir_cores.buffer(EROSION_DISTANCE_METERS, cap_style=3, join_style=2)

        # 4. SUBTRACT to find the rivers
        river_geoms = geom.difference(reservoir_geoms)

        # 5. Add results
        all_reservoirs.extend(flatten_geometries(reservoir_geoms))
        all_rivers.extend(flatten_geometries(river_geoms))

    # --- 3. Save outputs ---
    rivers_gdf = gpd.GeoDataFrame(geometry=all_rivers, crs=gdf.crs)
    reservoirs_gdf = gpd.GeoDataFrame(geometry=all_reservoirs, crs=gdf.crs)

    # Final cleanup of any tiny slivers using the same threshold
    rivers_gdf = rivers_gdf[rivers_gdf.geometry.area > MIN_AREA_THRESHOLD_METERS]
    reservoirs_gdf = reservoirs_gdf[reservoirs_gdf.geometry.area > MIN_AREA_THRESHOLD_METERS]

    # --- Save to NEW folders ---
    rivers_path = os.path.join(river_dir, f"rivers_{file}")
    reservoirs_path = os.path.join(reservoir_dir, f"reservoirs_{file}")

    if not rivers_gdf.empty:
        rivers_gdf.to_file(rivers_path)
    if not reservoirs_gdf.empty:
        reservoirs_gdf.to_file(reservoirs_path)

    print(f"✅ {file}: {len(rivers_gdf)} rivers, {len(reservoirs_gdf)} reservoirs extracted.")

print("🎯 Extraction complete — Rivers and Reservoirs saved in 'Rivers2' and 'Reservoirs2'.")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,8))
gdf.plot(ax=ax, color='lightgrey', edgecolor='black', linewidth=0.3)
gdf[gdf["Type"]=="River"].plot(ax=ax, color='blue', label='Rivers')
gdf[gdf["Type"]=="Reservoir"].plot(ax=ax, color='cyan', label='Reservoirs')
plt.legend()
plt.title(f"Rivers vs Reservoirs - {file}")
plt.show()


In [ ]:
import os
import geopandas as gpd
import matplotlib.pyplot as plt

base_dir = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second"
rivers_dir = os.path.join(base_dir, "Rivers")
reservoirs_dir = os.path.join(base_dir, "Reservoirs")
out_fig_dir = os.path.join(base_dir, "River_Reservoir_Comparisons")
os.makedirs(out_fig_dir, exist_ok=True)

# list files
river_files = sorted([f for f in os.listdir(rivers_dir) if f.endswith(".shp")])
reservoir_files = sorted([f for f in os.listdir(reservoirs_dir) if f.endswith(".shp")])

# helper to get core name (strip known prefixes)
def core_name(fname):
    name = os.path.splitext(fname)[0]
    # remove known prefixes if present
    for pref in ("rivers_", "river_", "reservoirs_", "reservoir_"):
        if name.startswith(pref):
            name = name[len(pref):]
            break
    return name

# build dicts mapping core -> filename
river_map = {core_name(f): f for f in river_files}
reservoir_map = {core_name(f): f for f in reservoir_files}

# find common cores
common_cores = sorted(set(river_map.keys()).intersection(reservoir_map.keys()))
only_rivers = sorted(set(river_map.keys()) - set(reservoir_map.keys()))
only_reservoirs = sorted(set(reservoir_map.keys()) - set(river_map.keys()))

print(f"🔗 Matched pairs: {len(common_cores)}")
if only_rivers:
    print(f"⚠️ Unmatched river files ({len(only_rivers)}): {only_rivers[:10]}{'...' if len(only_rivers)>10 else ''}")
if only_reservoirs:
    print(f"⚠️ Unmatched reservoir files ({len(only_reservoirs)}): {only_reservoirs[:10]}{'...' if len(only_reservoirs)>10 else ''}")

# Iterate matched pairs and plot side-by-side
for core in common_cores:
    river_fname = river_map[core]
    reservoir_fname = reservoir_map[core]
    river_path = os.path.join(rivers_dir, river_fname)
    reservoir_path = os.path.join(reservoirs_dir, reservoir_fname)

    try:
        gdf_river = gpd.read_file(river_path)
        gdf_res  = gpd.read_file(reservoir_path)
    except Exception as e:
        print(f"❌ Failed reading pair {core}: {e}")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    # If empty geodataframes, plot empty basemap
    if len(gdf_river) > 0:
        gdf_river.plot(ax=axes[0], color='blue', edgecolor='black', linewidth=0.3)
    else:
        axes[0].text(0.5, 0.5, "No river polygons", ha='center', va='center')

    if len(gdf_res) > 0:
        gdf_res.plot(ax=axes[1], color='cyan', edgecolor='black', linewidth=0.3)
    else:
        axes[1].text(0.5, 0.5, "No reservoir polygons", ha='center', va='center')

    axes[0].set_title(f"Rivers — {core}")
    axes[1].set_title(f"Reservoirs — {core}")
    for ax in axes:
        ax.axis('off')

    plt.suptitle(f"River vs Reservoir — {core}", fontsize=14)
    plt.tight_layout()

    # Optional: save figure (uncomment to enable)
    # out_png = os.path.join(out_fig_dir, f"compare_{core}.png")
    # plt.savefig(out_png, dpi=200, bbox_inches='tight')
    # print(f"💾 Saved: {out_png}")

    plt.show()

# print summary lists for manual inspection if needed
if only_rivers:
    print("\nList of river-only cores (no matching reservoir found):")
    for item in only_rivers:
        print(" -", item)
if only_reservoirs:
    print("\nList of reservoir-only cores (no matching river found):")
    for item in only_reservoirs:
        print(" -", item)


In [ ]:
import os
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt

# Define directories
parent_dir = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second"
vector_dir = os.path.join(parent_dir, "Water_Extracted_Shapes")
river_dir = os.path.join(parent_dir, "Rivers")
reservoir_dir = os.path.join(parent_dir, "Reservoirs")

# Create output folders if not exist
os.makedirs(river_dir, exist_ok=True)
os.makedirs(reservoir_dir, exist_ok=True)

# Helper: compute geometric shape metrics
def compute_shape_metrics(geom):
    try:
        area = geom.area
        perimeter = geom.length
        circularity = 4 * np.pi * area / (perimeter ** 2 + 1e-6)
        compactness = np.sqrt(4 * np.pi * area) / (perimeter + 1e-6)
        return circularity, compactness
    except Exception:
        return np.nan, np.nan

# Process each shapefile
for file in os.listdir(vector_dir):
    if file.endswith(".shp"):
        shp_path = os.path.join(vector_dir, file)
        print(f"\n🔍 Processing: {file}")

        # Load shapefile
        gdf = gpd.read_file(shp_path)
        gdf = gdf[gdf.is_valid & (gdf.geometry.type.isin(["Polygon", "MultiPolygon"]))]

        if len(gdf) == 0:
            print(f"⚠️ No valid polygons found in {file}")
            continue

        # Compute metrics
        gdf["circularity"], gdf["compactness"] = zip(*gdf.geometry.apply(compute_shape_metrics))

        # Remove NaN values
        gdf = gdf.dropna(subset=["compactness"])

        # Skip if not enough polygons to threshold
        if len(gdf) < 2:
            print(f"⚠️ Skipping {file} (not enough polygons)")
            continue

        # Apply Otsu threshold to compactness
        otsu_thresh = threshold_otsu(gdf["compactness"])
        print(f"📊 Otsu threshold (compactness): {otsu_thresh:.4f}")

        # Classify
        gdf["Type"] = np.where(gdf["compactness"] > otsu_thresh, "Reservoir", "River")

        # Split datasets
        rivers = gdf[gdf["Type"] == "River"]
        reservoirs = gdf[gdf["Type"] == "Reservoir"]

        # Define output paths
        rivers_path = os.path.join(river_dir, f"rivers_{file}")
        reservoirs_path = os.path.join(reservoir_dir, f"reservoirs_{file}")

        # Save shapefiles
        rivers.to_file(rivers_path)
        reservoirs.to_file(reservoirs_path)

        print(f"✅ Saved: {len(rivers)} rivers → {rivers_path}")
        print(f"✅ Saved: {len(reservoirs)} reservoirs → {reservoirs_path}")

        # Optional quick visualization
        fig, ax = plt.subplots(figsize=(10,8))
        gdf.plot(ax=ax, color='lightgrey', edgecolor='black', linewidth=0.3)
        rivers.plot(ax=ax, color='blue', label='Rivers')
        reservoirs.plot(ax=ax, color='cyan', label='Reservoirs')
        plt.legend()
        plt.title(f"Rivers vs Reservoirs — {file}")
        plt.show()


In [ ]:
import os
import geopandas as gpd
import matplotlib.pyplot as plt

base_dir = "/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second"
rivers_dir = os.path.join(base_dir, "Rivers2")
reservoirs_dir = os.path.join(base_dir, "Reservoirs2")
out_fig_dir = os.path.join(base_dir, "River_Reservoir_Comparisons")
os.makedirs(out_fig_dir, exist_ok=True)

# list files
river_files = sorted([f for f in os.listdir(rivers_dir) if f.endswith(".shp")])
reservoir_files = sorted([f for f in os.listdir(reservoirs_dir) if f.endswith(".shp")])

# helper to get core name (strip known prefixes)
def core_name(fname):
    name = os.path.splitext(fname)[0]
    # remove known prefixes if present
    for pref in ("rivers_", "river_", "reservoirs_", "reservoir_"):
        if name.startswith(pref):
            name = name[len(pref):]
            break
    return name

# build dicts mapping core -> filename
river_map = {core_name(f): f for f in river_files}
reservoir_map = {core_name(f): f for f in reservoir_files}

# find common cores
common_cores = sorted(set(river_map.keys()).intersection(reservoir_map.keys()))
only_rivers = sorted(set(river_map.keys()) - set(reservoir_map.keys()))
only_reservoirs = sorted(set(reservoir_map.keys()) - set(river_map.keys()))

print(f"🔗 Matched pairs: {len(common_cores)}")
if only_rivers:
    print(f"⚠️ Unmatched river files ({len(only_rivers)}): {only_rivers[:10]}{'...' if len(only_rivers)>10 else ''}")
if only_reservoirs:
    print(f"⚠️ Unmatched reservoir files ({len(only_reservoirs)}): {only_reservoirs[:10]}{'...' if len(only_reservoirs)>10 else ''}")

# Iterate matched pairs and plot side-by-side
for core in common_cores:
    river_fname = river_map[core]
    reservoir_fname = reservoir_map[core]
    river_path = os.path.join(rivers_dir, river_fname)
    reservoir_path = os.path.join(reservoirs_dir, reservoir_fname)

    try:
        gdf_river = gpd.read_file(river_path)
        gdf_res  = gpd.read_file(reservoir_path)
    except Exception as e:
        print(f"❌ Failed reading pair {core}: {e}")
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    # If empty geodataframes, plot empty basemap
    if len(gdf_river) > 0:
        gdf_river.plot(ax=axes[0], color='blue', edgecolor='black', linewidth=0.3)
    else:
        axes[0].text(0.5, 0.5, "No river polygons", ha='center', va='center')

    if len(gdf_res) > 0:
        gdf_res.plot(ax=axes[1], color='cyan', edgecolor='black', linewidth=0.3)
    else:
        axes[1].text(0.5, 0.5, "No reservoir polygons", ha='center', va='center')

    axes[0].set_title(f"Rivers — {core}")
    axes[1].set_title(f"Reservoirs — {core}")
    for ax in axes:
        ax.axis('off')

    plt.suptitle(f"River vs Reservoir — {core}", fontsize=14)
    plt.tight_layout()

    # Optional: save figure (uncomment to enable)
    # out_png = os.path.join(out_fig_dir, f"compare_{core}.png")
    # plt.savefig(out_png, dpi=200, bbox_inches='tight')
    # print(f"💾 Saved: {out_png}")

    plt.show()

# print summary lists for manual inspection if needed
if only_rivers:
    print("\nList of river-only cores (no matching reservoir found):")
    for item in only_rivers:
        print(" -", item)
if only_reservoirs:
    print("\nList of reservoir-only cores (no matching river found):")
    for item in only_reservoirs:
        print(" -", item)


In [ ]:
import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings

# --- Parts 1-4 of your script (loading data) are assumed to be here ---
# ...
# ... (Assuming reservoir_df and river_df are in memory)
# ...

# =================================================================
# 4.5. FILTER DATA BY YEAR (NEW)
# =================================================================
# Create a list of the specific years you want to analyze
years_to_include = [2018, 2020, 2022, 2024, 2025]

# Create new DataFrames that only contain data from those years
filtered_reservoir_df = reservoir_df[reservoir_df['year'].isin(years_to_include)].copy()
filtered_river_df = river_df[river_df['year'].isin(years_to_include)].copy()

print(f"--- Filtering data for years: {years_to_include} ---")
print(f"Using {len(filtered_reservoir_df)} of {len(reservoir_df)} reservoir records.")
print(f"Using {len(filtered_river_df)} of {len(river_df)} river records.")

# =================================================================
# 5. TEMPORAL MAPPING (PLOTTING - UPDATED)
# =================================================================

print("\n--- Generating Temporal Plots (Filtered) ---")

fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 10),
    sharex=True # Both plots will share the same X-axis
)

# --- Plot 1: Reservoir Area (Using FILTERED data) ---
ax1.plot(
    filtered_reservoir_df['plot_date'],
    filtered_reservoir_df['total_area_km2'],
    marker='o',
    linestyle='-',
    color='blue'
)
ax1.set_title(f'Total Reservoir Surface Area Over Time', fontsize=16)
ax1.set_ylabel('Area (km²)')
ax1.grid(True, linestyle='--', alpha=0.6)

# --- Plot 2: River Width (Using FILTERED data) ---
ax2.plot(
    filtered_river_df['plot_date'],
    filtered_river_df['avg_width_m'],
    marker='s',
    linestyle='--',
    color='green'
)
ax2.set_title(f'Average River Width Over Time', fontsize=16)
ax2.set_ylabel('Average Width (m)')
ax2.set_xlabel('Year')
ax2.grid(True, linestyle='--', alpha=0.6)

# Set X-axis ticks to be clean integers (using FILTERED data)
all_years = sorted(list(set(filtered_reservoir_df['year'].unique()) | set(filtered_river_df['year'].unique())))
# We need to create ticks for all years in the range, not just the ones with data
plot_ticks = np.arange(min(all_years), max(all_years) + 1)

ax2.set_xticks(plot_ticks)
ax2.set_xticklabels(plot_ticks.astype(int), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# =================================================================
# 5.5. FILTER DATA BY YEAR (NEW)
# =================================================================
# Create a list of the specific years you want to analyze
years_to_include = [2018, 2020, 2022, 2024, 2025]

# Create new DataFrames that only contain data from those years
filtered_reservoir_df = reservoir_df[reservoir_df['year'].isin(years_to_include)]
filtered_river_df = river_df[river_df['year'].isin(years_to_include)]

print(f"--- Filtering data for years: {years_to_include} ---")
print(f"Using {len(filtered_reservoir_df)} of {len(reservoir_df)} reservoir records.")
print(f"Using {len(filtered_river_df)} of {len(river_df)} river records.")


# =================================================================
# 6. SEASONAL COMPARISON (Box Plots)
# =================================================================
print("\n--- Generating Seasonal Comparison Plots (Filtered) ---")

# Define the logical order for our seasons
season_order = ['Winter', 'Summer', 'Monsoon']

# --- Create the figure ---
fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(12, 10)
)

# --- 1. Reservoir Area Box Plot (Using FILTERED data) ---
sns.boxplot(
    data=filtered_reservoir_df,  # <-- UPDATED
    x='season',
    y='total_area_km2',
    order=season_order,
    ax=ax1
)
# --- Updated Title ---
ax1.set_title(f'Seasonal Reservoir Area Distribution', fontsize=16)
ax1.set_xlabel('')
ax1.set_ylabel('Total Area (km²)')
ax1.grid(True, linestyle='--', alpha=0.6)


# --- 2. River Width Box Plot (Using FILTERED data) ---
sns.boxplot(
    data=filtered_river_df,  # <-- UPDATED
    x='season',
    y='avg_width_m',
    order=season_order,
    ax=ax2
)
# --- Updated Title ---
ax2.set_title(f'Seasonal River Width Distribution', fontsize=16)
ax2.set_xlabel('Season', fontsize=12)
ax2.set_ylabel('Average Width (m)')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# =================================================================
# 7. ALTERNATIVE: Seasonal Bar Chart (Averages)
# =================================================================
print("\n--- Generating Seasonal Average Bar Charts (Filtered) ---")

# Calculate the mean (average) for each season (Using FILTERED data)
reservoir_avg = filtered_reservoir_df.groupby('season')['total_area_km2'].mean().reindex(season_order) # <-- UPDATED
river_avg = filtered_river_df.groupby('season')['avg_width_m'].mean().reindex(season_order) # <-- UPDATED

# --- Create the figure ---
fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(10, 8)
)

# --- 1. Reservoir Area Bar Chart ---
reservoir_avg.plot(
    kind='bar',
    ax=ax1,
    color=['#add8e6', '#ffffe0', '#90ee90'], # Light blue, yellow, green
    edgecolor='black'
)
# --- Updated Title ---
ax1.set_title(f'Average Reservoir Area by Season', fontsize=16)
ax1.set_ylabel('Average Area (km²)')
ax1.set_xticklabels(season_order, rotation=0)

# --- 2. River Width Bar Chart ---
river_avg.plot(
    kind='bar',
    ax=ax2,
    color=['#add8e6', '#ffffe0', '#90ee90'],
    edgecolor='black'
)
# --- Updated Title ---
ax2.set_title(f'Average River Width by Season ', fontsize=16)
ax2.set_ylabel('Average Width (m)')
ax2.set_xticklabels(season_order, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Run this cell once to install the 'centerline' library
# The correct package name is 'centerline', not 'centerline-py'
!pip install centerline

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points
from scipy.spatial import distance
import warnings
warnings.filterwarnings('ignore')

# File paths
rivers_2020_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2020_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'
rivers_2025_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2025_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'

# Load shapefiles
print("Loading river data...")
rivers_2020 = gpd.read_file(rivers_2020_path)
rivers_2025 = gpd.read_file(rivers_2025_path)

print(f"2020 data: {len(rivers_2020)} features")
print(f"2025 data: {len(rivers_2025)} features")

# Function to identify main river (rightmost based on centroid)
def get_main_river(gdf, name):
    """Get the rightmost river based on centroid x-coordinate"""
    gdf['centroid_x'] = gdf.geometry.centroid.x
    main_river = gdf.loc[gdf['centroid_x'].idxmax()]
    print(f"\n{name} main river selected:")
    print(f"  Centroid X: {main_river['centroid_x']:.2f}")
    print(f"  Area: {main_river.geometry.area:.2f}")
    return main_river.geometry

# Extract main rivers (rightmost)
main_river_2020 = get_main_river(rivers_2020, "2020")
main_river_2025 = get_main_river(rivers_2025, "2025")

# Extract centerlines from polygons
def extract_centerline(polygon, num_points=500):
    """Extract approximate centerline from polygon using skeleton approach"""
    # Get boundary coordinates
    if polygon.geom_type == 'Polygon':
        coords = list(polygon.exterior.coords)
    else:  # MultiPolygon
        # Get largest polygon
        polygon = max(polygon.geoms, key=lambda p: p.area)
        coords = list(polygon.exterior.coords)

    # Simple centerline: use medial axis approximation
    # Sample points along the polygon
    boundary = polygon.boundary
    length = boundary.length

    points = []
    for i in range(num_points):
        distance_along = (i / num_points) * length
        point = boundary.interpolate(distance_along)
        points.append((point.x, point.y))

    return np.array(points)

print("\nExtracting centerlines...")
centerline_2020 = extract_centerline(main_river_2020)
centerline_2025 = extract_centerline(main_river_2025)

print(f"2020 centerline: {len(centerline_2020)} points")
print(f"2025 centerline: {len(centerline_2025)} points")

# Compute deviation metrics
def compute_hausdorff_distance(points1, points2):
    """Compute Hausdorff distance between two point sets"""
    return max(
        distance.directed_hausdorff(points1, points2)[0],
        distance.directed_hausdorff(points2, points1)[0]
    )

def compute_mean_deviation(points1, points2):
    """Compute mean minimum distance from points1 to points2"""
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist)
    return np.mean(distances), np.std(distances)

def compute_rmsd(points1, points2):
    """Compute Root Mean Square Distance between two point sets"""
    # For each point in set1, find nearest point in set2
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist ** 2)
    return np.sqrt(np.mean(distances))

# Calculate metrics
print("\nComputing deviation metrics...")
hausdorff_dist = compute_hausdorff_distance(centerline_2020, centerline_2025)
mean_dev_2020_to_2025, std_dev = compute_mean_deviation(centerline_2020, centerline_2025)
rmsd_value = compute_rmsd(centerline_2020, centerline_2025)

print(f"\n=== DEVIATION METRICS ===")
print(f"Hausdorff Distance: {hausdorff_dist:.2f} meters")
print(f"Mean Deviation (2020→2025): {mean_dev_2020_to_2025:.2f} ± {std_dev:.2f} meters")
print(f"RMSD: {rmsd_value:.2f} meters")

# Compute point-wise deviations for visualization
point_deviations = []
for p1 in centerline_2020:
    min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in centerline_2025])
    point_deviations.append(min_dist)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: Overlay of both rivers with main rivers highlighted
ax1 = axes[0, 0]
rivers_2020.plot(ax=ax1, color='lightblue', alpha=0.3, label='2020 All Rivers')
rivers_2025.plot(ax=ax1, color='lightcoral', alpha=0.3, label='2025 All Rivers')
gpd.GeoSeries([main_river_2020]).plot(ax=ax1, color='blue', alpha=0.6, linewidth=2, label='2020 Main River')
gpd.GeoSeries([main_river_2025]).plot(ax=ax1, color='red', alpha=0.6, linewidth=2, label='2025 Main River')
ax1.set_title('River Overlay: 2020 vs 2025 Monsoon', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.grid(True, alpha=0.3)

# Plot 2: Centerlines overlay
ax2 = axes[0, 1]
ax2.plot(centerline_2020[:, 0], centerline_2020[:, 1], 'b-', linewidth=2, label='2020 Centerline', alpha=0.7)
ax2.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'r-', linewidth=2, label='2025 Centerline', alpha=0.7)
ax2.set_title('Centerline Comparison', fontsize=14, fontweight='bold')
ax2.legend()
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.grid(True, alpha=0.3)
ax2.axis('equal')

# Plot 3: Deviation heatmap along 2020 centerline
ax3 = axes[1, 0]
scatter = ax3.scatter(centerline_2020[:, 0], centerline_2020[:, 1],
                     c=point_deviations, cmap='YlOrRd', s=30, alpha=0.8)
ax3.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'b--', linewidth=1, alpha=0.5, label='2025 Reference')
cbar = plt.colorbar(scatter, ax=ax3)
cbar.set_label('Deviation (meters)', rotation=270, labelpad=20)
ax3.set_title('Point-wise Deviation Map\n(from 2020 to nearest 2025 point)', fontsize=14, fontweight='bold')
ax3.legend()
ax3.set_xlabel('Longitude')
ax3.set_ylabel('Latitude')
ax3.grid(True, alpha=0.3)
ax3.axis('equal')

# Plot 4: Deviation statistics
ax4 = axes[1, 1]
ax4.hist(point_deviations, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax4.axvline(mean_dev_2020_to_2025, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_dev_2020_to_2025:.2f}m')
ax4.axvline(rmsd_value, color='green', linestyle='--', linewidth=2, label=f'RMSD: {rmsd_value:.2f}m')
ax4.set_title('Distribution of Deviations', fontsize=14, fontweight='bold')
ax4.set_xlabel('Deviation (meters)')
ax4.set_ylabel('Frequency')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add metrics text box
textstr = f'Hausdorff Distance: {hausdorff_dist:.2f}m\nMean Deviation: {mean_dev_2020_to_2025:.2f}m\nStd Deviation: {std_dev:.2f}m\nRMSD: {rmsd_value:.2f}m'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax4.text(0.65, 0.95, textstr, transform=ax4.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/river_meandering_analysis.png', dpi=300, bbox_inches='tight')
print("\nVisualization saved to: /content/drive/MyDrive/river_meandering_analysis.png")
plt.show()

# Create detailed report
print("\n" + "="*60)
print("RIVER MEANDERING ANALYSIS REPORT")
print("="*60)
print(f"\nAnalysis Period: 2020 Monsoon → 2025 Monsoon")
print(f"\nMain River Characteristics:")
print(f"  2020 River Area: {main_river_2020.area:.2f} sq meters")
print(f"  2025 River Area: {main_river_2025.area:.2f} sq meters")
print(f"  Area Change: {((main_river_2025.area - main_river_2020.area) / main_river_2020.area * 100):.2f}%")
print(f"\nMeandering Metrics:")
print(f"  Hausdorff Distance: {hausdorff_dist:.2f} m")
print(f"  Mean Deviation: {mean_dev_2020_to_2025:.2f} m")
print(f"  Standard Deviation: {std_dev:.2f} m")
print(f"  RMSD (Root Mean Square Distance): {rmsd_value:.2f} m")
print(f"  Maximum Deviation: {np.max(point_deviations):.2f} m")
print(f"  Minimum Deviation: {np.min(point_deviations):.2f} m")
print("\n" + "="*60)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points
from scipy.spatial import distance
import warnings
warnings.filterwarnings('ignore')

# File paths
rivers_2020_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2020_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'
rivers_2025_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2025_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'

# Load shapefiles
print("Loading river data...")
rivers_2020 = gpd.read_file(rivers_2020_path)
rivers_2025 = gpd.read_file(rivers_2025_path)

print(f"2020 data: {len(rivers_2020)} features")
print(f"2025 data: {len(rivers_2025)} features")

# Function to identify and merge main river parts (rightmost river system)
def get_main_river(gdf, name, x_threshold=340000):
    """
    Select all parts of the rightmost river system and merge them.
    Uses absolute X-coordinate threshold to ensure consistent selection across years.

    Parameters:
    -----------
    x_threshold : float
        Absolute X-coordinate threshold. Features with X > threshold are selected.
        Default: 340000 (adjust based on your data)
    """
    # Calculate centroids
    gdf['centroid_x'] = gdf.geometry.centroid.x
    gdf['centroid_y'] = gdf.geometry.centroid.y

    print(f"\n{name} - Identifying main river system:")
    print(f"  Total features: {len(gdf)}")
    print(f"  X-coordinate range: [{gdf['centroid_x'].min():.2f}, {gdf['centroid_x'].max():.2f}]")
    print(f"  Using X threshold: {x_threshold:.2f}")

    # Filter features on the right side using ABSOLUTE threshold
    main_river_parts = gdf[gdf['centroid_x'] > x_threshold].copy()

    print(f"  Parts belonging to main river: {len(main_river_parts)}")

    if len(main_river_parts) == 0:
        print(f"  WARNING: No features found with X > {x_threshold}!")
        print(f"  Suggestion: Lower the threshold. Max X in data: {gdf['centroid_x'].max():.2f}")
        # Fallback: use upper 30% of features
        sorted_gdf = gdf.sort_values('centroid_x', ascending=False)
        main_river_parts = sorted_gdf.head(max(1, int(len(gdf) * 0.3))).copy()
        print(f"  Fallback: Selected top {len(main_river_parts)} rightmost features")

    print(f"  Filtered X-coordinate range: [{main_river_parts['centroid_x'].min():.2f}, {main_river_parts['centroid_x'].max():.2f}]")

    # Merge all parts of the main river
    merged = main_river_parts.unary_union
    print(f"  Merged geometry type: {merged.geom_type}")
    print(f"  Total merged area: {merged.area:.2f}")

    return merged, main_river_parts

# IMPORTANT: Set the X threshold based on your data
# From the image, the main river appears to be around X > 340000
# Adjust this value if needed after checking the console output
X_THRESHOLD = 340000

# Extract main rivers (rightmost system, merged) using SAME threshold for both years
main_river_2020, main_parts_2020 = get_main_river(rivers_2020, "2020", X_THRESHOLD)
main_river_2025, main_parts_2025 = get_main_river(rivers_2025, "2025", X_THRESHOLD)

# Extract centerlines from polygons
def extract_centerline(polygon, num_points=500):
    """Extract approximate centerline from polygon using skeleton approach"""
    # Get boundary coordinates
    if polygon.geom_type == 'Polygon':
        coords = list(polygon.exterior.coords)
    else:  # MultiPolygon
        # Get largest polygon
        polygon = max(polygon.geoms, key=lambda p: p.area)
        coords = list(polygon.exterior.coords)

    # Simple centerline: use medial axis approximation
    # Sample points along the polygon
    boundary = polygon.boundary
    length = boundary.length

    points = []
    for i in range(num_points):
        distance_along = (i / num_points) * length
        point = boundary.interpolate(distance_along)
        points.append((point.x, point.y))

    return np.array(points)

print("\nExtracting centerlines...")
centerline_2020 = extract_centerline(main_river_2020)
centerline_2025 = extract_centerline(main_river_2025)

print(f"2020 centerline: {len(centerline_2020)} points")
print(f"2025 centerline: {len(centerline_2025)} points")

# Compute deviation metrics
def compute_hausdorff_distance(points1, points2):
    """Compute Hausdorff distance between two point sets"""
    return max(
        distance.directed_hausdorff(points1, points2)[0],
        distance.directed_hausdorff(points2, points1)[0]
    )

def compute_mean_deviation(points1, points2):
    """Compute mean minimum distance from points1 to points2"""
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist)
    return np.mean(distances), np.std(distances)

def compute_rmsd(points1, points2):
    """Compute Root Mean Square Distance between two point sets"""
    # For each point in set1, find nearest point in set2
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist ** 2)
    return np.sqrt(np.mean(distances))

# Calculate metrics
print("\nComputing deviation metrics...")
hausdorff_dist = compute_hausdorff_distance(centerline_2020, centerline_2025)
mean_dev_2020_to_2025, std_dev = compute_mean_deviation(centerline_2020, centerline_2025)
rmsd_value = compute_rmsd(centerline_2020, centerline_2025)

print(f"\n=== DEVIATION METRICS ===")
print(f"Hausdorff Distance: {hausdorff_dist:.2f} meters")
print(f"Mean Deviation (2020→2025): {mean_dev_2020_to_2025:.2f} ± {std_dev:.2f} meters")
print(f"RMSD: {rmsd_value:.2f} meters")

# Compute point-wise deviations for visualization
point_deviations = []
for p1 in centerline_2020:
    min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in centerline_2025])
    point_deviations.append(min_dist)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: Overlay of both rivers with main rivers highlighted
ax1 = axes[0, 0]
rivers_2020.plot(ax=ax1, color='lightgray', alpha=0.3, label='2020 All Rivers')
rivers_2025.plot(ax=ax1, color='lightgray', alpha=0.3, label='2025 All Rivers')
main_parts_2020.plot(ax=ax1, color='blue', alpha=0.6, linewidth=2, label='2020 Main River')
main_parts_2025.plot(ax=ax1, color='red', alpha=0.6, linewidth=2, label='2025 Main River')
ax1.set_title('River Overlay: 2020 vs 2025 Monsoon', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.grid(True, alpha=0.3)

# Plot 2: Centerlines overlay
ax2 = axes[0, 1]
ax2.plot(centerline_2020[:, 0], centerline_2020[:, 1], 'b-', linewidth=2, label='2020 Centerline', alpha=0.7)
ax2.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'r-', linewidth=2, label='2025 Centerline', alpha=0.7)
ax2.set_title('Centerline Comparison', fontsize=14, fontweight='bold')
ax2.legend()
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.grid(True, alpha=0.3)
ax2.axis('equal')

# Plot 3: Deviation heatmap along 2020 centerline
ax3 = axes[1, 0]
scatter = ax3.scatter(centerline_2020[:, 0], centerline_2020[:, 1],
                     c=point_deviations, cmap='YlOrRd', s=30, alpha=0.8)
ax3.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'b--', linewidth=1, alpha=0.5, label='2025 Reference')
cbar = plt.colorbar(scatter, ax=ax3)
cbar.set_label('Deviation (meters)', rotation=270, labelpad=20)
ax3.set_title('Point-wise Deviation Map\n(from 2020 to nearest 2025 point)', fontsize=14, fontweight='bold')
ax3.legend()
ax3.set_xlabel('Longitude')
ax3.set_ylabel('Latitude')
ax3.grid(True, alpha=0.3)
ax3.axis('equal')

# Plot 4: Deviation statistics
ax4 = axes[1, 1]
ax4.hist(point_deviations, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax4.axvline(mean_dev_2020_to_2025, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_dev_2020_to_2025:.2f}m')
ax4.axvline(rmsd_value, color='green', linestyle='--', linewidth=2, label=f'RMSD: {rmsd_value:.2f}m')
ax4.set_title('Distribution of Deviations', fontsize=14, fontweight='bold')
ax4.set_xlabel('Deviation (meters)')
ax4.set_ylabel('Frequency')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add metrics text box
textstr = f'Hausdorff Distance: {hausdorff_dist:.2f}m\nMean Deviation: {mean_dev_2020_to_2025:.2f}m\nStd Deviation: {std_dev:.2f}m\nRMSD: {rmsd_value:.2f}m'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax4.text(0.65, 0.95, textstr, transform=ax4.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/river_meandering_analysis.png', dpi=300, bbox_inches='tight')
print("\nVisualization saved to: /content/drive/MyDrive/river_meandering_analysis.png")
plt.show()

# Create detailed report
print("\n" + "="*60)
print("RIVER MEANDERING ANALYSIS REPORT")
print("="*60)
print(f"\nAnalysis Period: 2020 Monsoon → 2025 Monsoon")
print(f"\nMain River Characteristics:")
print(f"  2020 River Area: {main_river_2020.area:.2f} sq meters")
print(f"  2025 River Area: {main_river_2025.area:.2f} sq meters")
print(f"  Area Change: {((main_river_2025.area - main_river_2020.area) / main_river_2020.area * 100):.2f}%")
print(f"\nMeandering Metrics:")
print(f"  Hausdorff Distance: {hausdorff_dist:.2f} m")
print(f"  Mean Deviation: {mean_dev_2020_to_2025:.2f} m")
print(f"  Standard Deviation: {std_dev:.2f} m")
print(f"  RMSD (Root Mean Square Distance): {rmsd_value:.2f} m")
print(f"  Maximum Deviation: {np.max(point_deviations):.2f} m")
print(f"  Minimum Deviation: {np.min(point_deviations):.2f} m")
print("\n" + "="*60)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points
from scipy.spatial import distance
import warnings
warnings.filterwarnings('ignore')

# File paths
rivers_2020_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2020_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'
rivers_2025_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2025_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'

# Load shapefiles
print("Loading river data...")
rivers_2020 = gpd.read_file(rivers_2020_path)
rivers_2025 = gpd.read_file(rivers_2025_path)

print(f"2020 data: {len(rivers_2020)} features")
print(f"2025 data: {len(rivers_2025)} features")

# Function to identify and merge main river parts (rightmost river system)
def get_main_river(gdf, name, x_threshold=340000):
    """
    Select all parts of the rightmost river system and merge them.
    Uses absolute X-coordinate threshold to ensure consistent selection across years.

    Parameters:
    -----------
    x_threshold : float
        Absolute X-coordinate threshold. Features with X > threshold are selected.
        Default: 340000 (adjust based on your data)
    """
    # Calculate centroids
    gdf['centroid_x'] = gdf.geometry.centroid.x
    gdf['centroid_y'] = gdf.geometry.centroid.y

    print(f"\n{name} - Identifying main river system:")
    print(f"  Total features: {len(gdf)}")
    print(f"  X-coordinate range: [{gdf['centroid_x'].min():.2f}, {gdf['centroid_x'].max():.2f}]")
    print(f"  Using X threshold: {x_threshold:.2f}")

    # Filter features on the right side using ABSOLUTE threshold
    main_river_parts = gdf[gdf['centroid_x'] > x_threshold].copy()

    print(f"  Parts belonging to main river: {len(main_river_parts)}")

    if len(main_river_parts) == 0:
        print(f"  WARNING: No features found with X > {x_threshold}!")
        print(f"  Suggestion: Lower the threshold. Max X in data: {gdf['centroid_x'].max():.2f}")
        # Fallback: use upper 30% of features
        sorted_gdf = gdf.sort_values('centroid_x', ascending=False)
        main_river_parts = sorted_gdf.head(max(1, int(len(gdf) * 0.3))).copy()
        print(f"  Fallback: Selected top {len(main_river_parts)} rightmost features")

    print(f"  Filtered X-coordinate range: [{main_river_parts['centroid_x'].min():.2f}, {main_river_parts['centroid_x'].max():.2f}]")

    # Merge all parts of the main river
    merged = main_river_parts.unary_union
    print(f"  Merged geometry type: {merged.geom_type}")
    print(f"  Total merged area: {merged.area:.2f}")

    return merged, main_river_parts

# IMPORTANT: Set the X threshold based on your data
# From the image, the main river appears to be around X > 340000
# Adjust this value if needed after checking the console output
X_THRESHOLD = 340000

# Extract main rivers (rightmost system, merged) using SAME threshold for both years
main_river_2020, main_parts_2020 = get_main_river(rivers_2020, "2020", X_THRESHOLD)
main_river_2025, main_parts_2025 = get_main_river(rivers_2025, "2025", X_THRESHOLD)

# Extract centerlines from polygons
def extract_centerline(polygon, num_points=500):
    """Extract approximate centerline from polygon using skeleton approach"""
    # Get boundary coordinates
    if polygon.geom_type == 'Polygon':
        coords = list(polygon.exterior.coords)
    else:  # MultiPolygon
        # Get largest polygon
        polygon = max(polygon.geoms, key=lambda p: p.area)
        coords = list(polygon.exterior.coords)

    # Simple centerline: use medial axis approximation
    # Sample points along the polygon
    boundary = polygon.boundary
    length = boundary.length

    points = []
    for i in range(num_points):
        distance_along = (i / num_points) * length
        point = boundary.interpolate(distance_along)
        points.append((point.x, point.y))

    return np.array(points)

print("\nExtracting centerlines...")
centerline_2020 = extract_centerline(main_river_2020)
centerline_2025 = extract_centerline(main_river_2025)

print(f"2020 centerline: {len(centerline_2020)} points")
print(f"2025 centerline: {len(centerline_2025)} points")

# --- NEW FUNCTION ---
def compute_sinuosity(centerline_points):
    """
    Calculates the sinuosity of a river centerline.
    Sinuosity = Channel Length / Valley Length
    """
    if len(centerline_points) < 2:
        return np.nan # Not enough points to calculate

    # 1. Calculate Channel Length (actual path)
    # Convert numpy points to a Shapely LineString
    channel_line = LineString(centerline_points)
    channel_length = channel_line.length

    # 2. Calculate Valley Length (straight-line distance)
    start_point = centerline_points[0]
    end_point = centerline_points[-1]
    # Use numpy linalg.norm for simple Euclidean distance
    valley_length = np.linalg.norm(start_point - end_point)

    # 3. Calculate Sinuosity
    if valley_length == 0:
        return np.nan # Avoid division by zero

    sinuosity = channel_length / valley_length
    return sinuosity

# --- NEW: Calculate Sinuosity ---
print("\nComputing sinuosity...")
sinuosity_2020 = compute_sinuosity(centerline_2020)
sinuosity_2025 = compute_sinuosity(centerline_2025)

print(f"2020 Sinuosity Index: {sinuosity_2020:.3f}")
print(f"2025 Sinuosity Index: {sinuosity_2025:.3f}")

# Compute deviation metrics
def compute_hausdorff_distance(points1, points2):
    """Compute Hausdorff distance between two point sets"""
    return max(
        distance.directed_hausdorff(points1, points2)[0],
        distance.directed_hausdorff(points2, points1)[0]
    )

def compute_mean_deviation(points1, points2):
    """Compute mean minimum distance from points1 to points2"""
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist)
    return np.mean(distances), np.std(distances)

def compute_rmsd(points1, points2):
    """Compute Root Mean Square Distance between two point sets"""
    # For each point in set1, find nearest point in set2
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist ** 2)
    return np.sqrt(np.mean(distances))

# Calculate metrics
print("\nComputing deviation metrics...")
hausdorff_dist = compute_hausdorff_distance(centerline_2020, centerline_2025)
mean_dev_2020_to_2025, std_dev = compute_mean_deviation(centerline_2020, centerline_2025)
rmsd_value = compute_rmsd(centerline_2020, centerline_2025)

print(f"\n=== DEVIATION METRICS ===")
print(f"Hausdorff Distance: {hausdorff_dist:.2f} meters")
print(f"Mean Deviation (2020→2025): {mean_dev_2020_to_2025:.2f} ± {std_dev:.2f} meters")
print(f"RMSD: {rmsd_value:.2f} meters")

# Compute point-wise deviations for visualization
point_deviations = []
for p1 in centerline_2020:
    min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in centerline_2025])
    point_deviations.append(min_dist)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: Overlay of both rivers with main rivers highlighted
ax1 = axes[0, 0]
rivers_2020.plot(ax=ax1, color='lightgray', alpha=0.3, label='2020 All Rivers')
rivers_2025.plot(ax=ax1, color='lightgray', alpha=0.3, label='2025 All Rivers')
main_parts_2020.plot(ax=ax1, color='blue', alpha=0.6, linewidth=2, label='2020 Main River')
main_parts_2025.plot(ax=ax1, color='red', alpha=0.6, linewidth=2, label='2025 Main River')
ax1.set_title('River Overlay: 2020 vs 2025 Monsoon', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.grid(True, alpha=0.3)

# Plot 2: Centerlines overlay
ax2 = axes[0, 1]
ax2.plot(centerline_2020[:, 0], centerline_2020[:, 1], 'b-', linewidth=2, label='2020 Centerline', alpha=0.7)
ax2.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'r-', linewidth=2, label='2025 Centerline', alpha=0.7)
ax2.set_title('Centerline Comparison', fontsize=14, fontweight='bold')
ax2.legend()
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.grid(True, alpha=0.3)
ax2.axis('equal')

# Plot 3: Deviation heatmap along 2020 centerline
ax3 = axes[1, 0]
scatter = ax3.scatter(centerline_2020[:, 0], centerline_2020[:, 1],
                      c=point_deviations, cmap='YlOrRd', s=30, alpha=0.8)
ax3.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'b--', linewidth=1, alpha=0.5, label='2025 Reference')
cbar = plt.colorbar(scatter, ax=ax3)
cbar.set_label('Deviation (meters)', rotation=270, labelpad=20)
ax3.set_title('Point-wise Deviation Map\n(from 2020 to nearest 2025 point)', fontsize=14, fontweight='bold')
ax3.legend()
ax3.set_xlabel('Longitude')
ax3.set_ylabel('Latitude')
ax3.grid(True, alpha=0.3)
ax3.axis('equal')

# Plot 4: Deviation statistics
ax4 = axes[1, 1]
ax4.hist(point_deviations, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax4.axvline(mean_dev_2020_to_2025, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_dev_2020_to_2025:.2f}m')
ax4.axvline(rmsd_value, color='green', linestyle='--', linewidth=2, label=f'RMSD: {rmsd_value:.2f}m')
ax4.set_title('Distribution of Deviations', fontsize=14, fontweight='bold')
ax4.set_xlabel('Deviation (meters)')
ax4.set_ylabel('Frequency')
ax4.legend()
ax4.grid(True, alpha=0.3)

# --- UPDATED: Add metrics text box ---
# Added Sinuosity and section headers for clarity
textstr = (
    f'-- Deviation Metrics --\n'
    f'Hausdorff: {hausdorff_dist:.2f} m\n'
    f'Mean Dev: {mean_dev_2020_to_2025:.2f} m\n'
    f'RMSD: {rmsd_value:.2f} m\n'
    f'\n'
    f'-- Sinuosity Index --\n'
    f'2020: {sinuosity_2020:.3f}\n'
    f'2025: {sinuosity_2025:.3f}'
)

props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
# Adjusted position (0.60 -> 0.95) to better fit new text
ax4.text(0.60, 0.95, textstr, transform=ax4.transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/river_meandering_analysis.png', dpi=300, bbox_inches='tight')
print("\nVisualization saved to: /content/drive/MyDrive/river_meandering_analysis.png")
plt.show()

# --- UPDATED: Create detailed report ---
print("\n" + "="*60)
print("RIVER MEANDERING ANALYSIS REPORT")
print("="*60)
print(f"\nAnalysis Period: 2020 Monsoon → 2025 Monsoon")
print(f"\nMain River Characteristics:")
print(f"  2020 River Area: {main_river_2020.area:.2f} sq meters")
print(f"  2025 River Area: {main_river_2025.area:.2f} sq meters")
print(f"  Area Change: {((main_river_2025.area - main_river_2020.area) / main_river_2020.area * 100):.2f}%")

# Renamed section for clarity
print(f"\nDeviation Metrics (2020 vs 2025):")
print(f"  Hausdorff Distance: {hausdorff_dist:.2f} m")
print(f"  Mean Deviation: {mean_dev_2020_to_2025:.2f} m")
print(f"  Standard Deviation: {std_dev:.2f} m")
print(f"  RMSD (Root Mean Square Distance): {rmsd_value:.2f} m")
print(f"  Maximum Deviation: {np.max(point_deviations):.2f} m")
print(f"  Minimum Deviation: {np.min(point_deviations):.2f} m")

# Added new Sinuosity section
print(f"\nMeandering Metrics (Sinuosity):")
print(f"  2020 Sinuosity Index: {sinuosity_2020:.3f}")
print(f"  2025 Sinuosity Index: {sinuosity_2025:.3f}")
print(f"  Sinuosity Change: {sinuosity_2025 - sinuosity_2020:+.3f}")

print("\n" + "="*60)

In [ ]:
pip install centerline

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
from shapely.ops import nearest_points
from scipy.spatial import distance
from skimage.morphology import skeletonize, medial_axis
from skimage.draw import polygon as draw_polygon
from scipy.ndimage import distance_transform_edt
import warnings

warnings.filterwarnings('ignore')

# File paths
rivers_2020_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2020_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'
rivers_2025_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2025_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'

# Load shapefiles
print("Loading river data...")
rivers_2020 = gpd.read_file(rivers_2020_path)
rivers_2025 = gpd.read_file(rivers_2025_path)

print(f"2020 data: {len(rivers_2020)} features")
print(f"2025 data: {len(rivers_2025)} features")

# Function to identify and merge main river parts (rightmost river system)
def get_main_river(gdf, name, x_threshold=340000):
    """
    Select all parts of the rightmost river system and merge them.
    Uses absolute X-coordinate threshold to ensure consistent selection across years.
    """
    # Calculate centroids
    gdf['centroid_x'] = gdf.geometry.centroid.x
    gdf['centroid_y'] = gdf.geometry.centroid.y

    print(f"\n{name} - Identifying main river system:")
    print(f"  Total features: {len(gdf)}")
    print(f"  X-coordinate range: [{gdf['centroid_x'].min():.2f}, {gdf['centroid_x'].max():.2f}]")
    print(f"  Using X threshold: {x_threshold:.2f}")

    # Filter features on the right side using ABSOLUTE threshold
    main_river_parts = gdf[gdf['centroid_x'] > x_threshold].copy()

    print(f"  Parts belonging to main river: {len(main_river_parts)}")

    if len(main_river_parts) == 0:
        print(f"  WARNING: No features found with X > {x_threshold}!")
        print(f"  Suggestion: Lower the threshold. Max X in data: {gdf['centroid_x'].max():.2f}")
        # Fallback: use upper 30% of features
        sorted_gdf = gdf.sort_values('centroid_x', ascending=False)
        main_river_parts = sorted_gdf.head(max(1, int(len(gdf) * 0.3))).copy()
        print(f"  Fallback: Selected top {len(main_river_parts)} rightmost features")

    print(f"  Filtered X-coordinate range: [{main_river_parts['centroid_x'].min():.2f}, {main_river_parts['centroid_x'].max():.2f}]")

    # Merge all parts of the main river
    merged = main_river_parts.unary_union
    print(f"  Merged geometry type: {merged.geom_type}")
    print(f"  Total merged area: {merged.area:.2f}")

    return merged, main_river_parts

# IMPORTANT: Set the X threshold based on your data
X_THRESHOLD = 340000

# Extract main rivers (rightmost system, merged) using SAME threshold for both years
main_river_2020, main_parts_2020 = get_main_river(rivers_2020, "2020", X_THRESHOLD)
main_river_2025, main_parts_2025 = get_main_river(rivers_2025, "2025", X_THRESHOLD)


def extract_centerline_skeleton(geometry, resolution=10):
    """
    Extract centerline using morphological skeletonization.
    This method converts the polygon to a raster, computes the skeleton,
    and extracts the longest path.

    Args:
        geometry: Shapely Polygon or MultiPolygon
        resolution: Pixel resolution in meters (smaller = more detailed but slower)

    Returns:
        NumPy array of centerline coordinates
    """
    # Handle MultiPolygons
    if isinstance(geometry, MultiPolygon):
        print(f"  MultiPolygon with {len(geometry.geoms)} parts")
        geometry = max(geometry.geoms, key=lambda p: p.area)
        print(f"  Using largest polygon (area: {geometry.area:.2f})")

    if not isinstance(geometry, Polygon) or geometry.is_empty:
        print("  ERROR: Invalid geometry")
        return np.array([])

    try:
        # Get bounds and create raster grid
        minx, miny, maxx, maxy = geometry.bounds
        width = int((maxx - minx) / resolution) + 1
        height = int((maxy - miny) / resolution) + 1

        print(f"  Raster size: {width}x{height} pixels ({resolution}m resolution)")

        if width * height > 50000000:  # Limit to ~50M pixels
            print(f"  WARNING: Raster too large, increasing resolution")
            resolution = max(20, resolution * 2)
            width = int((maxx - minx) / resolution) + 1
            height = int((maxy - miny) / resolution) + 1
            print(f"  New size: {width}x{height} pixels ({resolution}m resolution)")

        # Create binary raster
        raster = np.zeros((height, width), dtype=bool)

        # Rasterize the polygon
        exterior_coords = np.array(geometry.exterior.coords)
        x_indices = ((exterior_coords[:, 0] - minx) / resolution).astype(int)
        y_indices = ((maxy - exterior_coords[:, 1]) / resolution).astype(int)

        # Clip indices to valid range
        x_indices = np.clip(x_indices, 0, width - 1)
        y_indices = np.clip(y_indices, 0, height - 1)

        rr, cc = draw_polygon(y_indices, x_indices, raster.shape)
        raster[rr, cc] = True

        print(f"  Filled pixels: {np.sum(raster)} ({np.sum(raster)/(width*height)*100:.1f}%)")

        if np.sum(raster) < 10:
            print("  ERROR: Too few pixels rasterized")
            return np.array([])

        # Compute skeleton using medial axis
        print("  Computing medial axis...")
        skeleton = medial_axis(raster)

        print(f"  Skeleton pixels: {np.sum(skeleton)}")

        if np.sum(skeleton) < 2:
            print("  ERROR: Skeleton too short")
            return np.array([])

        # Extract skeleton coordinates
        y_coords, x_coords = np.where(skeleton)

        # Convert back to original coordinate system
        real_x = x_coords * resolution + minx
        real_y = maxy - y_coords * resolution

        skeleton_points = np.column_stack([real_x, real_y])

        # Order points to form a continuous line
        ordered_points = order_skeleton_points(skeleton_points)

        # Smooth the centerline
        if len(ordered_points) > 5:
            ordered_points = smooth_centerline(ordered_points, window=5)

        print(f"  Extracted {len(ordered_points)} centerline points")

        return ordered_points

    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}")
        return np.array([])


def order_skeleton_points(points):
    """
    Order skeleton points to form a continuous path from one end to the other.
    Uses nearest neighbor ordering starting from an endpoint.
    """
    if len(points) < 2:
        return points

    # Find endpoints (points with only 1-2 neighbors within a threshold)
    from scipy.spatial import cKDTree
    tree = cKDTree(points)

    # Count neighbors within a reasonable distance
    distances = tree.query(points, k=min(5, len(points)))[0]
    avg_dist = np.median(distances[:, 1]) if len(points) > 1 else 10

    neighbor_counts = tree.query_ball_point(points, r=avg_dist * 2)
    neighbor_counts = [len(n) for n in neighbor_counts]

    # Start from point with fewest neighbors (likely an endpoint)
    start_idx = np.argmin(neighbor_counts)

    # Greedy path construction
    ordered = [points[start_idx]]
    remaining = set(range(len(points)))
    remaining.remove(start_idx)

    while remaining:
        current = ordered[-1]

        # Find nearest remaining point
        min_dist = float('inf')
        next_idx = None

        for idx in remaining:
            dist = np.linalg.norm(current - points[idx])
            if dist < min_dist:
                min_dist = dist
                next_idx = idx

        if next_idx is None:
            break

        ordered.append(points[next_idx])
        remaining.remove(next_idx)

    return np.array(ordered)


def smooth_centerline(points, window=5):
    """
    Smooth centerline using moving average.
    """
    if len(points) < window:
        return points

    # Pad the array
    padded_x = np.pad(points[:, 0], (window//2, window//2), mode='edge')
    padded_y = np.pad(points[:, 1], (window//2, window//2), mode='edge')

    # Apply moving average
    smoothed_x = np.convolve(padded_x, np.ones(window)/window, mode='valid')
    smoothed_y = np.convolve(padded_y, np.ones(window)/window, mode='valid')

    return np.column_stack([smoothed_x, smoothed_y])


print("\n" + "="*60)
print("EXTRACTING CENTERLINES (Skeleton Method)")
print("="*60)

print("\nExtracting 2020 centerline...")
centerline_2020 = extract_centerline_skeleton(main_river_2020, resolution=10)

print("\nExtracting 2025 centerline...")
centerline_2025 = extract_centerline_skeleton(main_river_2025, resolution=10)

# Check if centerline extraction was successful
if centerline_2020.size == 0 or centerline_2025.size == 0:
    print("\n" + "!"*60)
    print("FATAL ERROR: Centerline extraction failed!")
    print("!"*60)
    if centerline_2020.size == 0:
        print("2020 centerline extraction failed")
    if centerline_2025.size == 0:
        print("2025 centerline extraction failed")
    print("\nTrying alternative method with coarser resolution...")

    if centerline_2020.size == 0:
        centerline_2020 = extract_centerline_skeleton(main_river_2020, resolution=20)
    if centerline_2025.size == 0:
        centerline_2025 = extract_centerline_skeleton(main_river_2025, resolution=20)

    if centerline_2020.size == 0 or centerline_2025.size == 0:
        raise ValueError("Centerline extraction failed even with coarser resolution")

print(f"\n✓ 2020 centerline: {len(centerline_2020)} points")
print(f"✓ 2025 centerline: {len(centerline_2025)} points")

# Calculate Sinuosity
def compute_sinuosity(centerline_points):
    """
    Calculates the sinuosity of a river centerline.
    Sinuosity = Channel Length / Valley Length
    """
    if len(centerline_points) < 2:
        return np.nan

    # 1. Calculate Channel Length (sum of segment lengths)
    channel_line = LineString(centerline_points)
    channel_length = channel_line.length

    # 2. Calculate Valley Length (straight-line distance)
    start_point = centerline_points[0]
    end_point = centerline_points[-1]
    valley_length = np.linalg.norm(start_point - end_point)

    # 3. Calculate Sinuosity
    if valley_length == 0:
        return np.nan

    sinuosity = channel_length / valley_length
    return sinuosity, channel_length, valley_length

print("\n" + "="*60)
print("COMPUTING SINUOSITY")
print("="*60)
sinuosity_2020, channel_2020, valley_2020 = compute_sinuosity(centerline_2020)
sinuosity_2025, channel_2025, valley_2025 = compute_sinuosity(centerline_2025)

print(f"\n2020 River:")
print(f"  Channel Length: {channel_2020:.2f} m")
print(f"  Valley Length: {valley_2020:.2f} m")
print(f"  Sinuosity Index: {sinuosity_2020:.3f}")

print(f"\n2025 River:")
print(f"  Channel Length: {channel_2025:.2f} m")
print(f"  Valley Length: {valley_2025:.2f} m")
print(f"  Sinuosity Index: {sinuosity_2025:.3f}")

print(f"\nChange: {sinuosity_2025 - sinuosity_2020:+.3f}")

if abs(sinuosity_2025 - sinuosity_2020) < 0.01:
    interpretation = "negligible change"
elif sinuosity_2025 > sinuosity_2020:
    interpretation = "increased meandering"
else:
    interpretation = "decreased meandering (straightening)"

print(f"Interpretation: {interpretation}")

# Compute deviation metrics
def compute_hausdorff_distance(points1, points2):
    """Compute Hausdorff distance between two point sets"""
    return max(
        distance.directed_hausdorff(points1, points2)[0],
        distance.directed_hausdorff(points2, points1)[0]
    )

def compute_mean_deviation(points1, points2):
    """Compute mean minimum distance from points1 to points2"""
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist)
    return np.mean(distances), np.std(distances)

def compute_rmsd(points1, points2):
    """Compute Root Mean Square Distance between two point sets"""
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist ** 2)
    return np.sqrt(np.mean(distances))

# Calculate metrics
print("\n" + "="*60)
print("COMPUTING DEVIATION METRICS")
print("="*60)
hausdorff_dist = compute_hausdorff_distance(centerline_2020, centerline_2025)
mean_dev_2020_to_2025, std_dev = compute_mean_deviation(centerline_2020, centerline_2025)
rmsd_value = compute_rmsd(centerline_2020, centerline_2025)

print(f"Hausdorff Distance: {hausdorff_dist:.2f} meters")
print(f"Mean Deviation (2020→2025): {mean_dev_2020_to_2025:.2f} ± {std_dev:.2f} meters")
print(f"RMSD: {rmsd_value:.2f} meters")

# Compute point-wise deviations for visualization
point_deviations = []
for p1 in centerline_2020:
    min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in centerline_2025])
    point_deviations.append(min_dist)

# Create visualization
print("\n" + "="*60)
print("CREATING VISUALIZATIONS")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: Overlay of both rivers
ax1 = axes[0, 0]
rivers_2020.plot(ax=ax1, color='lightgray', alpha=0.3, label='2020 All Rivers')
rivers_2025.plot(ax=ax1, color='lightgray', alpha=0.3, label='2025 All Rivers')
main_parts_2020.plot(ax=ax1, color='blue', alpha=0.6, linewidth=2, label='2020 Main River')
main_parts_2025.plot(ax=ax1, color='red', alpha=0.6, linewidth=2, label='2025 Main River')
ax1.set_title('River Overlay: 2020 vs 2025 Monsoon', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.grid(True, alpha=0.3)

# Plot 2: Centerlines overlay
ax2 = axes[0, 1]
ax2.plot(centerline_2020[:, 0], centerline_2020[:, 1], 'b-', linewidth=2, label='2020 Centerline', alpha=0.7)
ax2.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'r-', linewidth=2, label='2025 Centerline', alpha=0.7)
ax2.set_title('Centerline Comparison (Skeleton Method)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.grid(True, alpha=0.3)
ax2.axis('equal')

# Plot 3: Deviation heatmap along 2020 centerline
ax3 = axes[1, 0]
scatter = ax3.scatter(centerline_2020[:, 0], centerline_2020[:, 1],
                      c=point_deviations, cmap='YlOrRd', s=30, alpha=0.8)
ax3.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'b--', linewidth=1, alpha=0.5, label='2025 Reference')
cbar = plt.colorbar(scatter, ax=ax3)
cbar.set_label('Deviation (meters)', rotation=270, labelpad=20)
ax3.set_title('Point-wise Deviation Map\n(from 2020 to nearest 2025 point)', fontsize=14, fontweight='bold')
ax3.legend()
ax3.set_xlabel('Longitude')
ax3.set_ylabel('Latitude')
ax3.grid(True, alpha=0.3)
ax3.axis('equal')

# Plot 4: Deviation statistics
ax4 = axes[1, 1]
ax4.hist(point_deviations, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax4.axvline(mean_dev_2020_to_2025, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_dev_2020_to_2025:.2f}m')
ax4.axvline(rmsd_value, color='green', linestyle='--', linewidth=2, label=f'RMSD: {rmsd_value:.2f}m')
ax4.set_title('Distribution of Deviations', fontsize=14, fontweight='bold')
ax4.set_xlabel('Deviation (meters)')
ax4.set_ylabel('Frequency')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add metrics text box
textstr = (
    f'-- Deviation Metrics --\n'
    f'Hausdorff: {hausdorff_dist:.2f} m\n'
    f'Mean Dev: {mean_dev_2020_to_2025:.2f} m\n'
    f'RMSD: {rmsd_value:.2f} m\n'
    f'\n'
    f'-- Sinuosity Index --\n'
    f'2020: {sinuosity_2020:.3f}\n'
    f'2025: {sinuosity_2025:.3f}\n'
    f'Change: {sinuosity_2025 - sinuosity_2020:+.3f}'
)
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax4.text(0.55, 0.95, textstr, transform=ax4.transAxes, fontsize=10,
         verticalalignment='top', bbox=props)

plt.tight_layout()
output_path = '/content/drive/MyDrive/river_meandering_analysis_skeleton.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"✓ Visualization saved to: {output_path}")
plt.show()

# Create detailed report
print("\n" + "="*60)
print("RIVER MEANDERING ANALYSIS REPORT")
print("="*60)
print(f"\nAnalysis Period: 2020 Monsoon → 2025 Monsoon")
print(f"Method: Morphological Skeleton Centerline Extraction")

print(f"\nMain River Characteristics:")
print(f"  2020 River Area: {main_river_2020.area:.2f} sq meters")
print(f"  2025 River Area: {main_river_2025.area:.2f} sq meters")
area_change_pct = ((main_river_2025.area - main_river_2020.area) / main_river_2020.area * 100)
print(f"  Area Change: {area_change_pct:+.2f}%")

print(f"\nCenterline Characteristics:")
print(f"  2020 Channel Length: {channel_2020:.2f} m")
print(f"  2020 Valley Length: {valley_2020:.2f} m")
print(f"  2025 Channel Length: {channel_2025:.2f} m")
print(f"  2025 Valley Length: {valley_2025:.2f} m")

print(f"\nDeviation Metrics (2020 vs 2025):")
print(f"  Hausdorff Distance: {hausdorff_dist:.2f} m")
print(f"  Mean Deviation: {mean_dev_2020_to_2025:.2f} m")
print(f"  Standard Deviation: {std_dev:.2f} m")
print(f"  RMSD (Root Mean Square Distance): {rmsd_value:.2f} m")
print(f"  Maximum Deviation: {np.max(point_deviations):.2f} m")
print(f"  Minimum Deviation: {np.min(point_deviations):.2f} m")

print(f"\nMeandering Metrics (Sinuosity):")
print(f"  2020 Sinuosity Index: {sinuosity_2020:.3f}")
print(f"  2025 Sinuosity Index: {sinuosity_2025:.3f}")
print(f"  Sinuosity Change: {sinuosity_2025 - sinuosity_2020:+.3f}")
print(f"  Interpretation: {interpretation}")

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points
from scipy.spatial import distance
import warnings
warnings.filterwarnings('ignore')

# File paths
rivers_2020_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2020_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'
rivers_2025_path = '/content/drive/MyDrive/GEE_Seasonal_Data_Nagarjuna_S2_Second/Rivers/rivers_2025_03_Monsoon_NDWI_MNDWI_WATERMASK.shp'

# Load shapefiles
print("Loading river data...")
rivers_2020 = gpd.read_file(rivers_2020_path)
rivers_2025 = gpd.read_file(rivers_2025_path)

print(f"2020 data: {len(rivers_2020)} features")
print(f"2025 data: {len(rivers_2025)} features")

# Function to identify and merge main river parts (rightmost river system)
def get_main_river(gdf, name, x_threshold=340000):
    """
    Select all parts of the rightmost river system and merge them.
    Uses absolute X-coordinate threshold to ensure consistent selection across years.

    Parameters:
    -----------
    x_threshold : float
        Absolute X-coordinate threshold. Features with X > threshold are selected.
        Default: 340000 (adjust based on your data)
    """
    # Calculate centroids
    gdf['centroid_x'] = gdf.geometry.centroid.x
    gdf['centroid_y'] = gdf.geometry.centroid.y

    print(f"\n{name} - Identifying main river system:")
    print(f"  Total features: {len(gdf)}")
    print(f"  X-coordinate range: [{gdf['centroid_x'].min():.2f}, {gdf['centroid_x'].max():.2f}]")
    print(f"  Using X threshold: {x_threshold:.2f}")

    # Filter features on the right side using ABSOLUTE threshold
    main_river_parts = gdf[gdf['centroid_x'] > x_threshold].copy()

    print(f"  Parts belonging to main river: {len(main_river_parts)}")

    if len(main_river_parts) == 0:
        print(f"  WARNING: No features found with X > {x_threshold}!")
        print(f"  Suggestion: Lower the threshold. Max X in data: {gdf['centroid_x'].max():.2f}")
        # Fallback: use upper 30% of features
        sorted_gdf = gdf.sort_values('centroid_x', ascending=False)
        main_river_parts = sorted_gdf.head(max(1, int(len(gdf) * 0.3))).copy()
        print(f"  Fallback: Selected top {len(main_river_parts)} rightmost features")

    print(f"  Filtered X-coordinate range: [{main_river_parts['centroid_x'].min():.2f}, {main_river_parts['centroid_x'].max():.2f}]")

    # Merge all parts of the main river
    merged = main_river_parts.unary_union
    print(f"  Merged geometry type: {merged.geom_type}")
    print(f"  Total merged area: {merged.area:.2f}")

    return merged, main_river_parts

# IMPORTANT: Set the X threshold based on your data
X_THRESHOLD = 340000

# Extract main rivers (rightmost system, merged) using SAME threshold for both years
main_river_2020, main_parts_2020 = get_main_river(rivers_2020, "2020", X_THRESHOLD)
main_river_2025, main_parts_2025 = get_main_river(rivers_2025, "2025", X_THRESHOLD)

# Extract centerlines from polygons
def extract_centerline(polygon, num_points=500):
    """Extract approximate centerline from polygon using skeleton approach"""
    # Get boundary coordinates
    if polygon.geom_type == 'Polygon':
        coords = list(polygon.exterior.coords)
    else:  # MultiPolygon
        # Get largest polygon
        polygon = max(polygon.geoms, key=lambda p: p.area)
        coords = list(polygon.exterior.coords)

    # Simple centerline: use medial axis approximation
    # Sample points along the polygon
    boundary = polygon.boundary
    length = boundary.length

    points = []
    for i in range(num_points):
        distance_along = (i / num_points) * length
        point = boundary.interpolate(distance_along)
        points.append((point.x, point.y))

    return np.array(points)

print("\nExtracting centerlines...")
centerline_2020 = extract_centerline(main_river_2020)
centerline_2025 = extract_centerline(main_river_2025)

print(f"2020 centerline: {len(centerline_2020)} points")
print(f"2025 centerline: {len(centerline_2025)} points")

# --- NEW FUNCTION: Local Migration Rate ---
def compute_local_migration_rate(centerline_old, centerline_new, time_interval_years=5):
    """
    Calculates local migration rates along the river centerline.

    Local Migration Rate = perpendicular distance moved / time interval

    Parameters:
    -----------
    centerline_old : numpy array
        Centerline coordinates from earlier time period
    centerline_new : numpy array
        Centerline coordinates from later time period
    time_interval_years : float
        Time between the two datasets in years (default: 5 for 2020-2025)

    Returns:
    --------
    migration_rates : numpy array
        Migration rate in meters/year for each point
    mean_rate : float
        Mean migration rate in meters/year
    max_rate : float
        Maximum migration rate in meters/year
    """
    if len(centerline_old) < 2 or len(centerline_new) < 2:
        return np.array([]), np.nan, np.nan

    migration_rates = []

    # For each point in old centerline, find nearest point in new centerline
    for p_old in centerline_old:
        # Calculate distances to all points in new centerline
        distances = np.linalg.norm(centerline_new - p_old, axis=1)
        min_distance = np.min(distances)

        # Convert distance to migration rate (meters per year)
        migration_rate = min_distance / time_interval_years
        migration_rates.append(migration_rate)

    migration_rates = np.array(migration_rates)
    mean_rate = np.mean(migration_rates)
    max_rate = np.max(migration_rates)

    return migration_rates, mean_rate, max_rate

# --- Calculate Local Migration Rate ---
print("\nComputing local migration rates...")
TIME_INTERVAL = 5  # years between 2020 and 2025

migration_rates, mean_migration_rate, max_migration_rate = compute_local_migration_rate(
    centerline_2020,
    centerline_2025,
    time_interval_years=TIME_INTERVAL
)

print(f"\n=== LOCAL MIGRATION RATE METRICS ===")
print(f"Mean Migration Rate: {mean_migration_rate:.2f} m/year")
print(f"Maximum Migration Rate: {max_migration_rate:.2f} m/year")
print(f"Minimum Migration Rate: {np.min(migration_rates):.2f} m/year")
print(f"Total Distance Migrated (Mean): {mean_migration_rate * TIME_INTERVAL:.2f} m over {TIME_INTERVAL} years")

# Compute deviation metrics
def compute_hausdorff_distance(points1, points2):
    """Compute Hausdorff distance between two point sets"""
    return max(
        distance.directed_hausdorff(points1, points2)[0],
        distance.directed_hausdorff(points2, points1)[0]
    )

def compute_mean_deviation(points1, points2):
    """Compute mean minimum distance from points1 to points2"""
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist)
    return np.mean(distances), np.std(distances)

def compute_rmsd(points1, points2):
    """Compute Root Mean Square Distance between two point sets"""
    distances = []
    for p1 in points1:
        min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in points2])
        distances.append(min_dist ** 2)
    return np.sqrt(np.mean(distances))

# Calculate metrics
print("\nComputing deviation metrics...")
hausdorff_dist = compute_hausdorff_distance(centerline_2020, centerline_2025)
mean_dev_2020_to_2025, std_dev = compute_mean_deviation(centerline_2020, centerline_2025)
rmsd_value = compute_rmsd(centerline_2020, centerline_2025)

print(f"\n=== DEVIATION METRICS ===")
print(f"Hausdorff Distance: {hausdorff_dist:.2f} meters")
print(f"Mean Deviation (2020→2025): {mean_dev_2020_to_2025:.2f} ± {std_dev:.2f} meters")
print(f"RMSD: {rmsd_value:.2f} meters")

# Compute point-wise deviations for visualization
point_deviations = []
for p1 in centerline_2020:
    min_dist = np.min([np.linalg.norm(p1 - p2) for p2 in centerline_2025])
    point_deviations.append(min_dist)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: Overlay of both rivers with main rivers highlighted
ax1 = axes[0, 0]
rivers_2020.plot(ax=ax1, color='lightgray', alpha=0.3, label='2020 All Rivers')
rivers_2025.plot(ax=ax1, color='lightgray', alpha=0.3, label='2025 All Rivers')
main_parts_2020.plot(ax=ax1, color='blue', alpha=0.6, linewidth=2, label='2020 Main River')
main_parts_2025.plot(ax=ax1, color='red', alpha=0.6, linewidth=2, label='2025 Main River')
ax1.set_title('River Overlay: 2020 vs 2025 Monsoon', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.grid(True, alpha=0.3)

# Plot 2: Centerlines overlay
ax2 = axes[0, 1]
ax2.plot(centerline_2020[:, 0], centerline_2020[:, 1], 'b-', linewidth=2, label='2020 Centerline', alpha=0.7)
ax2.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'r-', linewidth=2, label='2025 Centerline', alpha=0.7)
ax2.set_title('Centerline Comparison', fontsize=14, fontweight='bold')
ax2.legend()
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.grid(True, alpha=0.3)
ax2.axis('equal')

# Plot 3: Migration Rate Heatmap along 2020 centerline
ax3 = axes[1, 0]
scatter = ax3.scatter(centerline_2020[:, 0], centerline_2020[:, 1],
                      c=migration_rates, cmap='YlOrRd', s=30, alpha=0.8)
ax3.plot(centerline_2025[:, 0], centerline_2025[:, 1], 'b--', linewidth=1, alpha=0.5, label='2025 Reference')
cbar = plt.colorbar(scatter, ax=ax3)
cbar.set_label('Migration Rate (m/year)', rotation=270, labelpad=20)
ax3.set_title('Local Migration Rate Map\n(2020 to 2025)', fontsize=14, fontweight='bold')
ax3.legend()
ax3.set_xlabel('Longitude')
ax3.set_ylabel('Latitude')
ax3.grid(True, alpha=0.3)
ax3.axis('equal')

# Plot 4: Migration Rate Distribution
ax4 = axes[1, 1]
ax4.hist(migration_rates, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax4.axvline(mean_migration_rate, color='red', linestyle='--', linewidth=2,
            label=f'Mean: {mean_migration_rate:.2f} m/yr')
ax4.axvline(max_migration_rate, color='orange', linestyle='--', linewidth=2,
            label=f'Max: {max_migration_rate:.2f} m/yr')
ax4.set_title('Distribution of Migration Rates', fontsize=14, fontweight='bold')
ax4.set_xlabel('Migration Rate (meters/year)')
ax4.set_ylabel('Frequency')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add metrics text box
textstr = (
    f'-- Deviation Metrics --\n'
    f'Hausdorff: {hausdorff_dist:.2f} m\n'
    f'Mean Dev: {mean_dev_2020_to_2025:.2f} m\n'
    f'RMSD: {rmsd_value:.2f} m\n'
    f'\n'
    f'-- Migration Metrics --\n'
    f'Mean Rate: {mean_migration_rate:.2f} m/yr\n'
    f'Max Rate: {max_migration_rate:.2f} m/yr\n'
    f'Time Period: {TIME_INTERVAL} years'
)

props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax4.text(0.55, 0.95, textstr, transform=ax4.transAxes, fontsize=10,
         verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/river_migration_analysis.png', dpi=300, bbox_inches='tight')
print("\nVisualization saved to: /content/drive/MyDrive/river_migration_analysis.png")
plt.show()

# Create detailed report
print("\n" + "="*60)
print("RIVER MIGRATION ANALYSIS REPORT")
print("="*60)
print(f"\nAnalysis Period: 2020 Monsoon → 2025 Monsoon ({TIME_INTERVAL} years)")
print(f"\nMain River Characteristics:")
print(f"  2020 River Area: {main_river_2020.area:.2f} sq meters")
print(f"  2025 River Area: {main_river_2025.area:.2f} sq meters")
area_change_pct = ((main_river_2025.area - main_river_2020.area) / main_river_2020.area * 100)
print(f"  Area Change: {area_change_pct:+.2f}%")

print(f"\nDeviation Metrics (2020 vs 2025):")
print(f"  Hausdorff Distance: {hausdorff_dist:.2f} m")
print(f"  Mean Deviation: {mean_dev_2020_to_2025:.2f} m")
print(f"  Standard Deviation: {std_dev:.2f} m")
print(f"  RMSD (Root Mean Square Distance): {rmsd_value:.2f} m")
print(f"  Maximum Deviation: {np.max(point_deviations):.2f} m")
print(f"  Minimum Deviation: {np.min(point_deviations):.2f} m")

print(f"\nLocal Migration Rate Metrics:")
print(f"  Mean Migration Rate: {mean_migration_rate:.2f} m/year")
print(f"  Maximum Migration Rate: {max_migration_rate:.2f} m/year")
print(f"  Minimum Migration Rate: {np.min(migration_rates):.2f} m/year")
print(f"  Standard Deviation: {np.std(migration_rates):.2f} m/year")
print(f"  Total Migration (Mean): {mean_migration_rate * TIME_INTERVAL:.2f} m over {TIME_INTERVAL} years")
print(f"  Total Migration (Max): {max_migration_rate * TIME_INTERVAL:.2f} m over {TIME_INTERVAL} years")

# Interpretation
print(f"\nInterpretation:")
if mean_migration_rate < 5:
    print(f"  The river shows LOW migration activity ({mean_migration_rate:.2f} m/year)")
    print(f"  → Relatively stable channel position")
elif mean_migration_rate < 20:
    print(f"  The river shows MODERATE migration activity ({mean_migration_rate:.2f} m/year)")
    print(f"  → Normal meandering behavior")
else:
    print(f"  The river shows HIGH migration activity ({mean_migration_rate:.2f} m/year)")
    print(f"  → Active channel shifting and meandering")

print("\n" + "="*60)